In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score,recall_score

nltk.download("punkt_tab")
nltk.download("stopwords")
stop_words = set(stopwords.words('english'))

In [ ]:
def preprocess(tweet):
    assert isinstance(tweet, str), "Invalid type provided"

    # Convert to lowercase
    tweet = tweet.lower()

    # Remove URLs, mentions, hashtags, and non-alphabetic characters
    tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet, flags=re.MULTILINE)
    tweet = re.sub(r'@\w+', '', tweet)
    tweet = re.sub(r'#', '', tweet)
    tweet = re.sub(r'[^a-zA-Z\s]', '', tweet)

    # Tokenize
    tokens = word_tokenize(tweet)

    # Remove stopwords and short words
    tokens = [token for token in tokens if token not in stop_words and len(token) > 2]

    return ' '.join(tokens)

def train_model(
        model : torch.nn.Module,
        train_dataloader : torch.utils.data,
        test_dataloader : torch.utils.data,
        loss_fn : torch.nn,
        optimizer : torch.optim,
        accuracy_fn : sklearn.metrics,
        epochs : int,
        device : str,
        scheduler : torch.optim = None,
):
    
    model = model.to(device)
    for epoch in range(epochs):
        train_loss, train_acc = 0,0
        model.train()
        for (train_X, train_y) in train_dataloader:
            train_X, train_y = train_X.to(device), train_y.to(device)
            
            optimizer.zero_grad()
            train_logits = model(train_X).to(device)
            loss = loss_fn(train_logits, train_y)
            
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                train_preds = train_logits.argmax(dim=1)
                train_loss += loss.item()
                train_acc += accuracy_fn(train_y.cpu(), train_preds.cpu())

        train_loss /= len(train_dataloader)
        train_acc /= len(train_dataloader)
        if scheduler is not None:
            scheduler.step(train_loss)

        #Testing the model
        model.eval()
        test_loss, test_acc = 0,0
        with torch.no_grad:
            for (test_X, test_y) in test_dataloader:
                test_X, test_y = test_X.to(device), test_y.to(device)
                test_logits = model(test_X).to(device)
                test_preds = test_logits.argmax(dim=1)
                test_loss += loss_fn(test_logits, test_y).item()
                test_acc += accuracy_fn(test_y, test_preds)
            
        test_loss /= len(test_dataloader)
        test_acc /= len(test_dataloader)

        print(f"Epoch : {epoch+1} / {epochs}")
        print(f"Train Loss : {test_loss:.3f} | Train accuracy : {train_acc*100:.2f}%")
        print(f"Test Loss : {test_loss:.3f} | Test accuracy : {test_acc*100:.2f}%")
        print("-"*10)

def val_model(
        model : torch.nn.Module,
        val_dataloader : torch.utils.data,
        loss_fn : torch.nn,
        accuracy_fn : sklearn.metrics,
        device : str,
):
    
    model = model.to(device)
    model.eval()
    val_loss, val_acc = 0,0
    for (val_x, val_y) in val_dataloader:
        val_x, val_y = val_x.to(device), val_y.to(device)
        val_logits = model(val_x).to(device)
        val_preds = val_logits.argmax(dim=1)
        eval_loss += loss_fn(val_y, val_preds).item()
        eval_acc += accuracy_fn(val_y, val_preds)
    
    val_acc /= len(val_dataloader)
    val_loss /= len(val_dataloader)

    print(f"Eval loss : {val_loss:.3f} | Eval accuracy : {val_loss*100:.2f}%")
    print("-"*10)

class SA(nn.Module):
    def __init__(self,in_features, hidden_features, out_features):
        super().__init()    
        self.analyzer = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.BatchNorm1d(hidden_features),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_features,hidden_features//2),
            nn.BatchNorm1d(hidden_features//2),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_features//2, out_features)
        )

    def forward(self,x):
        return self.analyzer(x)

class SentimentDataset(Dataset):
    def __init__(self,x,y,scaler=None):
        super().__init__()
        if scaler is not None:
            self.features = torch.tensor(scaler.transform(x))
        else:
            self.features = torch.tensor(x)
        y = y - y.min()
        self.labels = torch.tensor(y)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx   ):
        return self.features[idx], self.labels[idx]
    

In [ ]:
data = pd.read_csv("./Datasets/twitter_data1.csv",names=["target", "id", "data", "flag", "user", "tweet"], encoding="latin-1")
data = data[['tweet', 'target']] #Keeping only the relevant columns
data = data.dropna()
len(data)

In [ ]:
tweets = data['tweet'].apply(preprocess)
labels = data['target']

#Vectorizer
vectorizer = TfidfVectorizer(max_features=1000,ngram_range=(1,2))
X = vectorizer.fit_transform(tweets)

#Scaling the data
scaler = StandardScaler()
X = scaler.fit_transfrom(X)

#Applying label encoding for the targets
le = LabelEncoder()
y = le.fit_transform(labels)

#Splitting data and creating datasets
train_X, temp_X, train_y, temp_y = train_test_split(X,y,test_size=0.1,random_state=42)
test_X, val_X, test_y, val_y = train_test_split(temp_X,temp_y,test_size=0.5,random_state=42)
 
train_dataset = SentimentDataset(train_X, train_y)
test_dataset = SentimentDataset(test_X, test_y)
val_dataset = SentimentDataset(val_X, val_y)

#Dataloaders
BATCH_SIZE = 32
train_dataloader = DataLoader(train_dataset,batch_size=BATCH_SIZE,drop_last=True, shuffle=True)
test_dataloader = DataLoader(test_dataloader,batch_size=BATCH_SIZE,drop_last=True)
val_dataloader = DataLoader(val_dataset,batch_size=BATCH_SIZE,drop_last=True)

#Training
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SA(
    in_features=X.shape[1],
    hidden_features=32,
    out_features=3,
).to(device)

# class_weights = [0]*len(set(labels))
# for x in labels:
#     if x == -1:
#         class_weights[0] += 1
#     elif x == 0:
#         class_weights[1] += 1
#     else:
#         class_weights[2] += 1

# for x in class_weights:
#     x = len(labels) / x

class_counts = Counter(data['category'])
max_samples = max(class_counts.values())
class_weights = torch.FloatTensor([max_samples / count for count in class_counts.values()])

loss_fn = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device)) 
optimizer = torch.optim.Adamax(params = model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3, 
)

train_model(model,train_dataloader, test_dataloader,loss_fn,optimizer,accuracy_score,20,device,scheduler)
val_model(model,val_dataloader,loss_fn,accuracy_score,device)

In [ ]:
for (train_X, train_y) in train_dataloader:
    print(y.size(0))